In [ ]:
!pip install -q xgboost imbalanced-learn pandas librosa seaborn
!apt-get install -y -q ffmpeg > /dev/null
import warnings
warnings.filterwarnings('ignore')
print("Kütüphaneler hazır!")

Kütüphaneler hazır!


In [ ]:
%%bash
rm -rf /content/dataset
rm -rf /content/archives
rm -rf /content/ornek_dinle
rm -f /content/*.tar.gz
echo "Temizlendi. Diskte kalan boş alan:"
df -h / | tail -1

Temizlendi. Diskte kalan boş alan:
overlay         113G   48G   66G  42% /


## 1. Veri ve Öznitelik Çıkarımı

In [ ]:
import numpy as np
import librosa
import os
import pandas as pd
from tqdm.notebook import tqdm

# Sabitler
SR         = 22050   # ortak örnekleme hızı (tutarlı kare hızı için)
N_MFCC     = 13      # MFCC katsayı sayısı
MAX_FRAMES = 216     # ~5 saniye (hop_length=512); diziler bu uzunluğa kırp ediyoruz

def extract_all_features(file_path):
    """Ses dosyasını TEK SEFERDE okur ve iki temsil döndürür:
       summary : (54,)               özet istatistik vektörü  -> Model 1 & 2
       seq     : (MAX_FRAMES, N_MFCC) zaman serili MFCC matrisi -> Model 3
    """
    try:
        y, sr = librosa.load(file_path, sr=SR)

        # ---------- (a) ZAMAN SERİLİ TEMSİL ----------
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)   # (N_MFCC, T)
        seq = mfccs.T                                             # (T, N_MFCC)
        if seq.shape[0] < MAX_FRAMES:
            seq = np.pad(seq, ((0, MAX_FRAMES - seq.shape[0]), (0, 0)))  # kısa -> sıfır doldur
        else:
            seq = seq[:MAX_FRAMES, :]                                    # uzun -> kırp

        # ---------- (b) ÖZET İSTATİSTİK TEMSİLİ ----------
        mfccs_mean = np.mean(mfccs, axis=1)
        mfccs_std  = np.std(mfccs, axis=1)

        zcr = librosa.feature.zero_crossing_rate(y=y)
        zcr_mean, zcr_std = np.mean(zcr), np.std(zcr)

        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        rolloff_mean = np.mean(rolloff)

        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_mean = np.mean(chroma, axis=1)
        chroma_std  = np.std(chroma, axis=1)

        rms = librosa.feature.rms(y=y)
        rms_mean = np.mean(rms)

        summary = np.concatenate((
            mfccs_mean, mfccs_std,
            [zcr_mean, zcr_std],
            [rolloff_mean],
            chroma_mean, chroma_std,
            [rms_mean]
        ))
        return summary, seq.astype(np.float32)
    except Exception:
        return None, None

def extract_features_from_file(file_path):
    """Ses dosyasını işleyerek ağ mimarileri için iki farklı öznitelik temsili çıkarır:
    1) summary : (54,) boyutunda istatistiksel özet vektörü (Geleneksel Baseline Modeli - XGBoost için)
    2) seq     : (MAX_FRAMES, N_MFCC) boyutunda zaman serisi matrisi (Önerilen 1D-CNN Mimarisi için)
    """
    summary, _ = extract_all_features(file_path)
    return summary

In [ ]:
import requests, tarfile, os

# Mozilla Common Voice API anahtarı
API_KEY = "BURAYA_KENDI_ANAHTARINIZI_YAZIN"

# Sınıflar ve hedeflenen örnek sayıları
diller = [
    ("cmn2g7uu701fqo1072r5na25l", "ARABIC",  1000),
    ("cmndapwry02jnmh07dyo46mot", "ENGLISH", 1000),
    ("cmn5zugst00w3nv07upovf2bg", "FRENCH",  1000),
]

def get_download_url(dataset_id):
    """API üzerinden dinamik indirme linki alır."""
    r = requests.post(
        f"https://mozilladatacollective.com/api/datasets/{dataset_id}/download",
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    )
    return r.json()["downloadUrl"]

def stream_process(dataset_id, label, limit=1000):
    """
    Bellek optimizasyonu için 'on-the-fly' (akış halinde) veri işleme.
    Her dosya için özet ve zaman serisi matrislerini tek seferde çıkarır.
    """
    print(f"\n=== {label}: Akış halinde indiriliyor ve işleniyor ===")
    url = get_download_url(dataset_id)
    data_list, seq_list = [], []
    count = 0

    # stream=True ile arşivi RAM'i şişirmeden parça parça çekiyoruz
    resp = requests.get(url, stream=True)
    resp.raw.decode_content = True

    with tarfile.open(fileobj=resp.raw, mode="r|gz") as tar:
        pbar = tqdm(total=limit, desc=label)
        for member in tar:
            if count >= limit:
                break

            if "clips/" in member.name and member.name.endswith(".mp3"):
                f = tar.extractfile(member)
                if f is None:
                    continue

                # Sesi temp dosyaya yaz ve özellikleri çıkarıyoruz
                tmp = "/tmp/_cur.mp3"
                with open(tmp, "wb") as out:
                    out.write(f.read())

                summary, seq = extract_all_features(tmp)

                if summary is not None:
                    data_list.append(np.append(summary, label))
                    seq_list.append(seq)
                    count += 1
                    pbar.update(1)
        pbar.close()
    resp.close()
    print(f"{label}: Toplam {count} dosya işlendi.")
    return data_list, seq_list

all_data, all_seqs = [], []

for dataset_id, label, lim in diller:
    d, s = stream_process(dataset_id, label, limit=lim)
    all_data.extend(d)
    all_seqs.extend(s)

columns = [f'mfcc_mean_{i}' for i in range(13)] + [f'mfcc_std_{i}' for i in range(13)] + \
          ['zcr_mean', 'zcr_std', 'rolloff_mean'] + \
          [f'chroma_mean_{i}' for i in range(12)] + [f'chroma_std_{i}' for i in range(12)] + \
          ['rms_mean', 'Accent']

# 1. TEMSİL: XGBoost (Baseline) için 2D DataFrame
df = pd.DataFrame(all_data, columns=columns)
for col in df.columns[:-1]:
    df[col] = pd.to_numeric(df[col])

# 2. TEMSİL: 1D-CNN (Sinir Ağı) için 3 Boyutlu Tensör -> (N, 216, 13)
X_seq = np.stack(all_seqs)

print(f"\nÖzet veri seti (XGBoost için) : {df.shape[0]} satır × {df.shape[1]} sütun")
print(f"Zaman serisi (1D-CNN için)  : {X_seq.shape}")


=== ARABIC: Akış halinde indiriliyor ve işleniyor ===


ARABIC:   0%|          | 0/1000 [00:00<?, ?it/s]

ARABIC: Toplam 1000 dosya işlendi.

=== ENGLISH: Akış halinde indiriliyor ve işleniyor ===


ENGLISH:   0%|          | 0/1000 [00:00<?, ?it/s]

ENGLISH: Toplam 1000 dosya işlendi.

=== FRENCH: Akış halinde indiriliyor ve işleniyor ===


FRENCH:   0%|          | 0/1000 [00:00<?, ?it/s]

FRENCH: Toplam 1000 dosya işlendi.

Özet veri seti (XGBoost için) : 3000 satır × 55 sütun
Zaman serisi (1D-CNN için)  : (3000, 216, 13)


## 2. Model 1 — XGBoost (Geleneksel Baseline)

Üç modelin adil karşılaştırılabilmesi için veri tek bir ortak indeks kümesi üzerinden bölünür: her model birebir aynı test örnekleriyle değerlendirilir.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

X = df.drop('Accent', axis=1).values
y = df['Accent'].values

# Sınıf etiketlerini sayısala çeviriyoruz (ARABIC: 0, ENGLISH: 1, FRENCH: 2)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# -ORTAK VERİ BÖLÜMLEME (DATA SPLITTING)-
# Modellerin adil karşılaştırılması için XGBoost ve 1D-CNN aynı test setini görmelidir.
# Bu yüzden veriyi doğrudan bölmek yerine, indeksler üzerinden bölümlendirme yapıyoruz.
indices = np.arange(len(y_encoded))
idx_train, idx_test = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y_encoded
)

X_train, X_test = X[idx_train], X[idx_test]
y_train, y_test = y_encoded[idx_train], y_encoded[idx_test]

X_seq_train, X_seq_test = X_seq[idx_train], X_seq[idx_test]

#-SMOTE İLE VERİ DENGELEME-
# İndirme sırasındaki hatalı mp3'ler nedeniyle oluşabilecek ufak dengesizlikleri gideriyoruz.
# Veri sızıntısını (Data Leakage) önlemek için SMOTE SADECE eğitim (train) setine uygulanır.
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("XGBoost (Baseline) Modeli Eğitiliyor...")
# Hiperparametreler: Aşırı öğrenmeyi (Overfitting) engellemek için düşük learning_rate
# ve sınırlandırılmış max_depth kullanılmıştır.
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)
model.fit(X_train_res, y_train_res)

y_pred_xgb = model.predict(X_test)
acc_xgb = accuracy_score(y_test, y_pred_xgb)

print("\n" + "="*55)
print("MODEL 1 — XGBoost (Baseline) Sonuçları")
print("="*55)
print(classification_report(le.inverse_transform(y_test), le.inverse_transform(y_pred_xgb)))
print(f"Genel Doğruluk (Accuracy): % {acc_xgb * 100:.2f}")

## 3. Model 2 — 1D-CNN


In [ ]:
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv1D, BatchNormalization, MaxPooling1D,
                                     GlobalAveragePooling1D, Dense, Dropout)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.random.set_seed(42)

# -VERİ ÖLÇEKLENDİRME VE TENSÖR DÖNÜŞÜMÜ-
# Veri sızıntısını (Data Leakage) önlemek için scaler SADECE eğitim setine fit edilir.
scaler = StandardScaler()
# Keras Conv1D katmanı 3 boyutlu girdi beklediği için veriyi (Örnek, Öznitelik, 1) şeklinde reshape ediyoruz.
X_train_sum = scaler.fit_transform(X_train_res).reshape(-1, X.shape[1], 1)
X_test_sum  = scaler.transform(X_test).reshape(-1, X.shape[1], 1)

n_classes = len(le.classes_)

# -1D-CNN MİMARİSİ-
# 1. Filtre sayısı derinleştikçe (64->128->256) artırılarak karmaşık örüntülerin yakalanması hedeflenmiştir.
# 2. BatchNormalization ile eğitim stabilize edilmiş ve hızlandırılmıştır.
# 3. Flatten yerine GlobalAveragePooling1D kullanılarak parametre sayısı düşürülmüş ve Overfitting engellenmiştir.
cnn_sum = Sequential([
    Conv1D(64, kernel_size=5, padding='same', activation='relu', input_shape=(X.shape[1], 1)),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Conv1D(128, kernel_size=3, padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Conv1D(256, kernel_size=3, padding='same', activation='relu'),
    BatchNormalization(),

    # Boyut indirgeme ve düzenlileştirme katmanları
    GlobalAveragePooling1D(),
    Dense(128, activation='relu'),
    Dropout(0.4), # Küçük veri setinde modelin ezberlemesini önlemek için
    Dense(n_classes, activation='softmax')
], name="CNN_Ozet")

# Modeli derleme (Adam optimizatörü ve Çoklu Sınıflandırma kayıp fonksiyonu ile)
cnn_sum.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
cnn_sum.summary()

#-EĞİTİM OPTİMİZASYONU (CALLBACKS)-
callbacks_sum = [
    # val_loss 15 epoch boyunca iyileşmezse eğitimi durdur ve en başarılı ağırlıkları (weights) geri yükle
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    # val_loss platoya (düzlüğe) girdiğinde öğrenme oranını (learning rate) kısarak ince ayar yap
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-5, verbose=1)
]

print("\n1D-CNN Modeli Eğitiliyor...")
history_sum = cnn_sum.fit(
    X_train_sum, y_train_res,
    validation_split=0.15,
    epochs=150,
    batch_size=32,
    callbacks=callbacks_sum,
    verbose=1
)

# -TEST VE BAŞARIM ÖLÇÜMÜ-
y_pred_probs = cnn_sum.predict(X_test_sum, verbose=0)
y_pred_sum = np.argmax(y_pred_probs, axis=1)

acc_sum = accuracy_score(y_test, y_pred_sum)
print(f"\nMODEL 2 — 1D-CNN Test Accuracy: % {acc_sum * 100:.2f}")

## 4. Model 3 — Zaman Serili 1D-CNN


In [ ]:
# == 3D TENSÖR İÇİN KANAL BAZLI STANDARDİZASYON ===
# Zaman serilerinde (Time-Series) standart scaler 3D yapıyı bozabileceği için
# istatistikler manuel olarak sadece eğitim verisinden (veri sızıntısını önlemek için) hesaplanır.
# 1e-8 değeri, sıfıra bölünme hatasını (zero-division error) önlemek için eklenmiştir.
seq_mean = X_seq_train.mean(axis=(0, 1), keepdims=True)
seq_std  = X_seq_train.std(axis=(0, 1), keepdims=True) + 1e-8

X_seq_train_n = (X_seq_train - seq_mean) / seq_std
X_seq_test_n  = (X_seq_test  - seq_mean) / seq_std

print(f"Girdi şekli: {X_seq_train_n.shape}  (Örnek, Zaman Adımı, MFCC Kanalı)")

# -ZAMAN SERİSİ 1D-CNN MİMARİSİ (Kontrollü Deney)-
# Bir önceki "Özet CNN" ile aynı omurga (backbone) kullanılarak kontrollü bir deney tasarlanmıştır.
# Amaç: Performans artışının mimariden değil, "Girdi Temsilinden" (zaman serisi) kaynaklandığını kanıtlamak.
# Zaman serisi verisi daha yüksek boyutlu olduğu için katman aralarına Dropout(0.25) eklenmiştir.
cnn_seq = Sequential([
    Conv1D(64, kernel_size=5, padding='same', activation='relu',
           input_shape=(MAX_FRAMES, N_MFCC)),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.25),

    Conv1D(128, kernel_size=5, padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.25),

    Conv1D(256, kernel_size=3, padding='same', activation='relu'),
    BatchNormalization(),
    GlobalAveragePooling1D(),

    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(n_classes, activation='softmax')
], name="CNN_ZamanSerili")

cnn_seq.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
cnn_seq.summary()

# -EĞİTİM OPTİMİZASYONU-
callbacks_seq = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-5, verbose=1)
]

print("\nZaman Serili 1D-CNN Modeli Eğitiliyor...")
# SMOTE interpolasyon yaptığı için zaman matrislerinde anlamsız "ara sesler (gürültü)" üretir.Bu yüzden kullanmadık.
history_seq = cnn_seq.fit(
    X_seq_train_n, y_train,
    validation_split=0.15,
    epochs=150,
    batch_size=32,
    callbacks=callbacks_seq,
    verbose=1
)

# -TEST VE BAŞARIM ÖLÇÜMÜ-
y_pred_probs_seq = cnn_seq.predict(X_seq_test_n, verbose=0)
y_pred_seq = np.argmax(y_pred_probs_seq, axis=1)

acc_seq = accuracy_score(y_test, y_pred_seq)
print(f"\nMODEL 3 — Zaman Serili CNN Test Accuracy: % {acc_seq * 100:.2f}")

## 5. Eğitim Süreçleri (Epoch–Loss / Epoch–Accuracy)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="darkgrid", context="talk", rc={
    "figure.facecolor": "#1c1e26", "axes.facecolor": "#252836",
    "axes.edgecolor": "#44475a", "grid.color": "#3b3f51",
    "text.color": "#f8f8f2", "axes.labelcolor": "#f8f8f2",
    "xtick.color": "#c5c8d8", "ytick.color": "#c5c8d8"
})
C_TRAIN, C_VAL, C_BEST = "#8be9fd", "#ff79c6", "#f1fa8c"

def plot_history(ax_loss, ax_acc, history, title):
    hist = pd.DataFrame(history.history)
    hist["Epoch"] = hist.index + 1
    for ax, m, vm, ylab in ((ax_loss, "loss", "val_loss", "Loss"),
                            (ax_acc, "accuracy", "val_accuracy", "Accuracy")):
        sns.lineplot(data=hist, x="Epoch", y=m,  ax=ax, linewidth=2.5, color=C_TRAIN, label="Eğitim")
        sns.lineplot(data=hist, x="Epoch", y=vm, ax=ax, linewidth=2.5, color=C_VAL,   label="Doğrulama")
        ax.set_ylabel(ylab)
        ax.set_title(f"{title} — {ylab}", fontweight="bold", fontsize=14, pad=10)
        ax.legend(frameon=False, fontsize=11)
    best = int(hist["val_loss"].idxmin()) + 1
    for ax in (ax_loss, ax_acc):
        ax.axvline(best, color=C_BEST, linestyle="--", linewidth=1.5, alpha=0.8)
    ax_loss.annotate(f"en iyi epoch: {best}", xy=(best, hist["val_loss"].min()),
                     xytext=(10, 15), textcoords="offset points", color=C_BEST, fontsize=11)

# -1D-CNN EĞİTİM SÜREÇLERİNİN (MODEL 2 VE 3) KARŞILAŞTIRILMASI-
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
plot_history(axes[0, 0], axes[0, 1], history_sum, "Model 2: Özet-CNN")
plot_history(axes[1, 0], axes[1, 1], history_seq, "Model 3: Zaman Serili CNN")
fig.suptitle("1D-CNN Eğitim Süreçleri", fontweight="bold", y=1.0)
plt.tight_layout()
plt.savefig("/content/egitim_egrileri.png", dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

## 6. Karşılaştırmalı Sonuçlar

Üç model aynı test kümesi üzerinde değerlendirilir: classification report, accuracy özeti ve yan yana confusion matrix'ler.

In [ ]:
from sklearn.metrics import confusion_matrix

labels = le.classes_

# -MODELLERİN DEĞERLENDİRME LİSTESİ-
results = [
    ("Model 1: XGBoost (Baseline)",      y_pred_xgb, acc_xgb, "mako"),
    ("Model 2: 1D-CNN (Özet)",           y_pred_sum, acc_sum, "viridis"),
    ("Model 3: Zaman Serili CNN (Önerilen)", y_pred_seq, acc_seq, "rocket"),
]

for name, y_pred_i, acc_i, _ in results:
    print("="*60)
    print(name)
    print("="*60)
    print(classification_report(le.inverse_transform(y_test), le.inverse_transform(y_pred_i)))
    print(f"Accuracy: % {acc_i*100:.2f}\n")

# -Accuracy özet grafiği-
fig, ax = plt.subplots(figsize=(10, 4.5))
names = [r[0].split(":")[0] + "\n" + r[0].split(":")[1].strip() for r in results]
accs  = [r[2]*100 for r in results]
bars = sns.barplot(x=names, y=accs, hue=names, ax=ax,
                   palette=["#6272a4", "#8be9fd", "#ff79c6"], legend=False)
for i, v in enumerate(accs):
    ax.text(i, v + 0.6, f"%{v:.2f}", ha="center", fontweight="bold", color="#f8f8f2", fontsize=13)
ax.set_ylim(0, max(accs)*1.18)
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Modellerin Test Başarımı", fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig("/content/accuracy_karsilastirma.png", dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

# -Yan yana Confusion Matrix'ler (ham sayı + satır bazında %)-
fig, axes = plt.subplots(1, 3, figsize=(21, 6.2))
for ax, (name, y_pred_i, acc_i, cmap) in zip(axes, results):
    cm = confusion_matrix(y_test, y_pred_i)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    annot = np.array([[f"{cm[i, j]}\n(%{cm_norm[i, j]*100:.0f})"
                       for j in range(len(labels))] for i in range(len(labels))])
    sns.heatmap(cm_norm, annot=annot, fmt="", cmap=cmap, vmin=0, vmax=1,
                xticklabels=labels, yticklabels=labels, ax=ax,
                linewidths=1.5, linecolor="#1c1e26",
                cbar_kws={"label": "Oran"}, annot_kws={"fontsize": 12})
    ax.set_title(f"{name}\nAccuracy: %{acc_i*100:.2f}", fontweight="bold", fontsize=13, pad=12)
    ax.set_xlabel("Tahmin Edilen Sınıf")
    ax.set_ylabel("Gerçek Sınıf")
fig.suptitle("Confusion Matrix Karşılaştırması", fontweight="bold", y=1.04)
plt.tight_layout()
plt.savefig("/content/confusion_matrix_karsilastirma.png", dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

## 7. Canlı Demo — Gradio

Kullanıcı `gr.Radio` ile üç modelden birini seçer; kaydedilen/yüklenen ses, seçilen modelin eğitimde kullanılan ön işleme adımlarından aynen geçirilerek sınıflandırılır.

In [16]:
!pip install -q gradio

import gradio as gr

MODEL_XGB = "Geleneksel Baseline: XGBoost"
MODEL_SUM = "1D-CNN (Özet Öznitelikler)"
MODEL_SEQ = "Önerilen Yaklaşım: Zaman Serili 1D-CNN"

def tahmin_et(ses_dosyasi, model_secimi):
    if ses_dosyasi is None:
        return "Lütfen önce bir ses kaydedin veya yükleyin."

    summary, seq = extract_all_features(ses_dosyasi)
    if summary is None:
        return "Ses işlenemedi, lütfen başka bir kayıt deneyin."

    if model_secimi == MODEL_SEQ:
        girdi = ((seq[np.newaxis, :, :] - seq_mean) / seq_std)      # eğitimdekiyle aynı normalizasyon
        olasiliklar = cnn_seq.predict(girdi, verbose=0)[0]
    elif model_secimi == MODEL_SUM:
        girdi = scaler.transform(summary.reshape(1, -1)).reshape(1, -1, 1)
        olasiliklar = cnn_sum.predict(girdi, verbose=0)[0]
    else:
        olasiliklar = model.predict_proba(summary.reshape(1, -1))[0]

    diller = le.classes_
    return {diller[i]: float(olasiliklar[i]) for i in range(len(diller))}

arayuz = gr.Interface(
    fn=tahmin_et,
    inputs=[
        gr.Audio(sources=["microphone", "upload"], type="filepath",
                 label="Ses kaydedin ya da bir dosya yükleyin"),
        gr.Radio(choices=[MODEL_XGB, MODEL_SUM, MODEL_SEQ], value=MODEL_SEQ,
                 label="Model Seçimi",
                 info="Tahminde kullanılacak modeli seçin")
    ],
    outputs=gr.Label(num_top_classes=3, label="Tahmin edilen dil/aksan"),
    title="🎙️ Aksan / Dil Tanıma — Üç Modelin Karşılaştırmalı Demosu",
    description="5 saniyelik bir ses kaydedin ve modeli seçin. Model dilin Arapça, İngilizce mi yoksa Fransızca mı olduğunu tahmin eder. NOT: Model yalnızca bu üç dili bilir; başka bir dil konuşulursa yine bu üçünden birini seçer."
)

arayuz.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3e9d0ca1f9584893ae.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
